In [ ]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import sys
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
from strat import STRAT

from dema_strat import (
  compare_benchmark,
  compare_table,
  db_connect,
  global_style,
  grid_tuning,
  load_data,
)
from exits_builder import build_be_exits
from monte_carlo import propfirm_sim_leverage
from params_finder import neighborhood_analysis, radius_diagnostics
from sizer import sl_based_sizer
from visualizer import (
  combinations_compare_visualizer,
  portfolio_perf_visualizer,
  sensitivity_check_visualizer,
  trades_visualizer,
)

sys.path.insert(0, str(Path.cwd().parent))
from constants import CASH

# CONFIG
TABLE_NAME = 'f0'
TABLE_NAME_OOS = f"{TABLE_NAME}_OOS"
DATA_PATH = '~/market_data/XAUUSD/XAUUSD_M15.csv'
TARGET = 'profit_factor'
RISK_PER_TRADE = 20
RISK_PCT = CASH / RISK_PER_TRADE 

PARAMS = {
    "ema_period": {"type": "INTEGER", "data": range(5, 15), "default": 9},
    "smoothing_period": {"type": "INTEGER", "data": range(6, 16), "default": 12},
    "use_vwap": {"type": "BOOL", "data": [True, False], "default": True},
    "atr_period": {"type": "INTEGER", "data": range(14, 22, 2), "default": 18},
    "sl_mult": {"type": "REAL", "data": range(1,3), "default": 1},
    "rr_ratio": {"type": "REAL", "data": [0.25, 0.5, 1.0, 1.5, 2.0], "default": 1},
}

PARAM_COLS = list(PARAMS)
PARAMS_GRID = list(product(*(v["data"] for v in PARAMS.values())))
PARAMS_DEFAULT = {
    k: v.get("default", v["data"][0]) 
    for k, v in PARAMS.items()
}

TOP_N = max(100, int(len(PARAMS_GRID) / 10))

conn, cursor = db_connect()

def run_backtest(data, params = PARAMS_DEFAULT):
  # build the strat
  strat = STRAT(data, params)

  rr_ratio = params.get("rr_ratio", 2)
  sl_mult = params.get("sl_mult", 1)

  # exits and sizer
  long_exits, short_exits, sl_dist = build_be_exits(data, strat.signals, rr_ratio=rr_ratio, sl_mult=sl_mult)
  sizer = sl_based_sizer(risk_per_trade=RISK_PER_TRADE, sl_dists=sl_dist)

  portfolio = strat._run_backtest(long_exits, short_exits, size=sizer, size_type="amount")
  return portfolio, strat

ohlcv_data = load_data(DATA_PATH)

## Grid tuning

In [ ]:
grid_tuning(params=PARAMS, table_name=TABLE_NAME, data=ohlcv_data, run_backtest=run_backtest, replace_table=True)
# grid_tuning(params=PARAMS, table_name=f"{TABLE_NAME}_OOS", data=df_oos, run_backtest=run_backtest, replace_table=True)

## Combinations compare for TOP vs All combinations

In [ ]:
def get_query(limit = True):
    return f"""
        SELECT *
        FROM {TABLE_NAME}
        ORDER BY {TARGET} DESC
        {f'LIMIT {TOP_N}' if limit else ''}
    """

top_df = pd.read_sql_query(get_query(), conn)
all_df = pd.read_sql_query(get_query(limit = False), conn)

combinations_compare_visualizer(
  first_data=top_df,
  second_data=all_df,
  chart_title=f"TOP {TOP_N} vs All combinations",
  first_name="TOP 200",
  second_name="All combinations"
)

## Parameter neighborhood analysis 

In [ ]:
df = pd.read_sql_query(f"""
    SELECT *
    FROM {TABLE_NAME}
""", conn)

report, best_r = radius_diagnostics(df, PARAMS, TARGET, radii=(1, 2, 3, 4))
best_combination = neighborhood_analysis(df, PARAMS, TARGET=TARGET, radius=best_r)

## Sensitivity check

In [ ]:
df = pd.read_sql_query(f"""
    SELECT *
    FROM {TABLE_NAME}
""", conn)

sensitivity_check_visualizer(df, PARAM_COLS=PARAM_COLS, TARGET=TARGET)

## Benchmark

In [ ]:
best_combination_param = best_combination[PARAM_COLS].to_dict()
print(best_combination_param)

portfolio, strat = run_backtest(ohlcv_data, params=best_combination_param)

compare_benchmark(
  data=ohlcv_data,
  portfolio=portfolio 
)

## Trade and Portfolio performance

In [ ]:
trades_visualizer(data=ohlcv_data, signals=strat.signals, portfolio=portfolio, N = 500)
portfolio_perf_visualizer(portfolio)

## Calc the EV for propfirm trading (FTMO)

In [ ]:
port_value = portfolio.value()
log_returns = np.log(port_value / port_value.shift(1)).dropna()

# Estimate bars per day from log returns frequency (average count per calendar day)
bars_per_day = int(log_returns.groupby(log_returns.index.date).size().mean())

CAPITAL = 10_000
params = {
    "tp" : 1500 / CAPITAL,
    "sl" : 1000 / CAPITAL,
    "daily_sl" : 0.05,
    "trailing_sl" : False,
    "bars_per_day" : bars_per_day,
    "days_per_month" : 21,
    "n_days" : 252,
    "capital" : CAPITAL,
    "challenge_cost" : 100,
    "fixed_cost_month" : 1,
    "payout_pct" : 0.80,
    "bankroll" : 2_000,
}

params["bars_per_month"] = params["days_per_month"] * params["bars_per_day"]
params["n_bars"] = params["n_days"] * params["bars_per_day"]

leverage_list = range(1, 11)

df = propfirm_sim_leverage(
  log_returns=log_returns,
  leverage_list=leverage_list,
  params=params,
)
global_style(df)

## Calc the EV for propfirm trading (TOPSTEPX)

In [ ]:
port_value = portfolio.value()
log_returns = np.log(port_value / port_value.shift(1)).dropna()

bars_per_day = int(log_returns.groupby(log_returns.index.date).size().mean())

CAPITAL = 50_000
params = {
    "tp" : 3000 / CAPITAL,
    "sl" : 2000 / CAPITAL,
    "daily_sl" : 3000 / CAPITAL,
    "trailing_sl" : True,
    "bars_per_day" : bars_per_day,
    "days_per_month" : 21,
    "n_days" : 252,
    "capital" : CAPITAL,
    "challenge_cost" : 50,
    "fixed_cost_month" : 50,
    "payout_pct" : 0.90,
    "bankroll" : 2_000,
}

params["bars_per_month"] = params["days_per_month"] * params["bars_per_day"]
params["n_bars"] = params["n_days"] * params["bars_per_day"]

leverage_list = range(1, 11)

df = propfirm_sim_leverage(
  log_returns=log_returns,
  leverage_list=leverage_list,
  params=params,
)

global_style(df)